# MulCon – Explainable Check-Worthiness via Rationality Label Representations

**Key insight**: The check-worthiness decision is made *from* the rationality label representations — not independently. This is what makes the system *explainable*: the CW prediction is grounded in the same embeddings that produce R1–R6.

## Correct Pipeline (from CheckMate paper)

```
Claim Text
  └─► BERT Encoder  →  token embeddings (B, seq_len, 768)
          └─► MulCon MultiAttBlock(U, r, r)  →  g_i (B, 6, D)   [one embedding per rationality label]
                  ├─► 6 × Classifier f_c^j   →  R1…R6 ∈ {0,1}  [rationality predictions]
                  ├─► Aggregate g_i           →  h (B, D)         [pooled rationality context]
                  └─► CW Head(h)              →  CW ∈ {0,1}       [check-worthiness from rationality]
```

## Loss
```
L = L_CW(from rationality repr)  +  L_BCE(R1–R6)  +  γ · L_con
```

## Expected CSV Columns
- `text` / `claim` / `tweet` – the claim text
- `check_worthy` (or `label` / `cw`) – 1 = check-worthy, 0 = not
- `R1`…`R6` – rationality labels (0/1); non-CW rows **may still have active rationality labels**

## Fix Applied
**Bug**: Original code zeroed all rationality labels for non-CW rows (`df.loc[cw==0, LABEL_COLS] = 0`),
destroying real signal and preventing the model from learning that a claim can be factual/public-interest
yet still not check-worthy.

**Fix**:
1. `load_data` – removed forced zeroing; true labels preserved for all rows
2. `compute_loss` – rationality BCE trained on all rows with true labels
3. `compute_loss` – contrastive loss uses any row with ≥1 active rationality label (not just CW rows)
4. `evaluate` – rationality metrics computed on all rows with ≥1 active rationality label

## 1. Install Dependencies

In [ ]:
!pip install transformers scikit-learn pandas numpy torch matplotlib --quiet

## 2. Imports

In [ ]:
import os, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR, StepLR

from transformers import BertTokenizer, BertModel
from sklearn.metrics import (
    f1_score, accuracy_score, classification_report,
    confusion_matrix, precision_score, recall_score, roc_auc_score
)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

## 3. Configuration

In [ ]:
# ─────────────────────────────────────────────────────────────
# FILE PATHS  ← update these
# ─────────────────────────────────────────────────────────────
TRAIN_FILE = 'train_data.csv'
VAL_FILE   = 'val_data.csv'
TEST_FILE  = 'test_data.csv'

# ─────────────────────────────────────────────────────────────
# COLUMN NAMES
# ─────────────────────────────────────────────────────────────
TEXT_COL   = 'claim'
CW_COL     = 'bin_label'
LABEL_COLS = [
    'verifiable_factual_claim',   # R1
    'false_info',                 # R2
    'general_public_interest',    # R3
    'harmful',                    # R4
    'fact_checker_interest',      # R5
    'govt_interest'               # R6
]

# ─────────────────────────────────────────────────────────────
# MODEL HYPERPARAMETERS
# ─────────────────────────────────────────────────────────────
BERT_MODEL_NAME = 'bert-base-uncased'
MAX_SEQ_LEN     = 128
NUM_RAT_LABELS  = 6
EMBED_DIM       = 768
LABEL_EMBED_DIM = 256
PROJ_DIM        = 128
NUM_ATTN_HEADS  = 4
DROPOUT         = 0.3        # raised from 0.25 — better regularisation on small data

# ─────────────────────────────────────────────────────────────
# LOSS WEIGHTS
# ─────────────────────────────────────────────────────────────
LAMBDA_CW   = 1.0
LAMBDA_RAT  = 1.0
GAMMA       = 0.1
TEMPERATURE = 0.2

# ─────────────────────────────────────────────────────────────
# TRAINING HYPERPARAMETERS
#
# Overfitting fixes applied vs original:
#   STEP1_LR:         2e-4  → 3e-5   (was 10x too high for BERT fine-tuning)
#   STEP1_BATCH_SIZE: 32    → 16     (more gradient updates, better generalisation)
#   STEP1_EPOCHS:     10    → 20     (larger budget; early stopping controls length)
#   STEP2_LR:         1e-4  → 5e-5   (raised from 1e-5 which was too low for SCL)
#   STEP2_BATCH_SIZE: 16    → 32     (larger batches = more SCL positive pairs)
#   STEP2_EPOCHS:     10    → 5      (Step 2 is fine-tuning only, not full training)
#   PATIENCE:         new   = 3      (early stopping added to both steps)
#   BERT frozen in Step 2           (prevents SGD from degrading AdamW-trained BERT)
# ─────────────────────────────────────────────────────────────
STEP1_LR         = 3e-5
STEP1_BATCH_SIZE = 16
STEP1_EPOCHS     = 20       # early stopping will cut this short

STEP2_LR         = 5e-5
STEP2_BATCH_SIZE = 32       # larger batches needed for contrastive positive pairs
STEP2_EPOCHS     = 5

PATIENCE     = 3
WEIGHT_DECAY = 1e-4
THRESHOLD    = 0.5

OUTPUT_DIR = './checkpoints'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Configuration ready.')
print(f'  STEP1: LR={STEP1_LR} | BS={STEP1_BATCH_SIZE} | epochs={STEP1_EPOCHS} | patience={PATIENCE}')
print(f'  STEP2: LR={STEP2_LR} | BS={STEP2_BATCH_SIZE} | epochs={STEP2_EPOCHS} | BERT frozen')
print(f'  DROPOUT={DROPOUT} | WEIGHT_DECAY={WEIGHT_DECAY}')


## 4. Data Loading

**Fix**: Removed the forced zeroing of rationality labels for non-CW rows.
Non-CW rows may genuinely have some active rationality labels (e.g. a claim can be
factual but not check-worthy). Zeroing them was destroying real training signal.

In [ ]:
def load_data(filepath):
    df = pd.read_csv(filepath)

    print(f"\nLoaded: {filepath} ({len(df)} rows)")
    print(f"Columns: {list(df.columns)}")

    # ── Validate required columns ──────────────────────────────────────────
    required_cols = [TEXT_COL, CW_COL] + LABEL_COLS
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    # ── Drop missing values ────────────────────────────────────────────────
    df = df.dropna(subset=required_cols)

    # ── Type conversion ────────────────────────────────────────────────────
    df[CW_COL]     = df[CW_COL].astype(int)
    df[LABEL_COLS] = df[LABEL_COLS].astype(int)

    # ── FIX: Do NOT zero rationality labels for non-CW rows ───────────────
    # Non-CW claims can still be factual, public-interest, etc.
    # The original code forced:
    #   df.loc[df[CW_COL] == 0, LABEL_COLS] = 0   ← REMOVED
    # That destroyed real signal and prevented the model from learning
    # the nuance between "has rationality factors" vs "is check-worthy".

    # ── Extract data ───────────────────────────────────────────────────────
    texts      = df[TEXT_COL].astype(str).tolist()
    cw_labels  = df[CW_COL].values.astype(np.float32)
    rat_labels = df[LABEL_COLS].values.astype(np.float32)

    # ── Stats ──────────────────────────────────────────────────────────────
    pos = int(cw_labels.sum())
    print(f"CW: {pos} | Non-CW: {len(df) - pos}")

    # Show how many non-CW rows still have active rationality labels
    non_cw_mask = cw_labels == 0
    non_cw_with_rat = int((rat_labels[non_cw_mask].sum(axis=1) > 0).sum())
    print(f"Non-CW rows with ≥1 active rationality label: {non_cw_with_rat}")

    return texts, cw_labels, rat_labels


# ── Load datasets ──────────────────────────────────────────────────────────────
train_texts, train_cw, train_rat = load_data(TRAIN_FILE)
val_texts,   val_cw,   val_rat   = load_data(VAL_FILE)
test_texts,  test_cw,  test_rat  = load_data(TEST_FILE)

## 5. Dataset & DataLoaders

In [ ]:
class CheckItDataset(Dataset):
    def __init__(self, texts, cw_labels, rat_labels, tokenizer, max_len=MAX_SEQ_LEN):
        self.texts      = texts
        self.cw_labels  = cw_labels
        self.rat_labels = rat_labels
        self.tokenizer  = tokenizer
        self.max_len    = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'cw_label':       torch.tensor(self.cw_labels[idx],  dtype=torch.float32),
            'rat_labels':     torch.tensor(self.rat_labels[idx], dtype=torch.float32),
        }


tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)

train_ds = CheckItDataset(train_texts, train_cw, train_rat, tokenizer)
val_ds   = CheckItDataset(val_texts,   val_cw,   val_rat,   tokenizer)
test_ds  = CheckItDataset(test_texts,  test_cw,  test_rat,  tokenizer)

def make_loader(ds, bs, shuffle=False):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle,
                      num_workers=0, pin_memory=(DEVICE.type=='cuda'))

train_loader_s1 = make_loader(train_ds, STEP1_BATCH_SIZE, shuffle=True)
train_loader_s2 = make_loader(train_ds, STEP2_BATCH_SIZE, shuffle=True)
val_loader      = make_loader(val_ds,   STEP1_BATCH_SIZE)
test_loader     = make_loader(test_ds,  STEP1_BATCH_SIZE)

print('DataLoaders ready.')
print(f'  S1 train batches/epoch: {len(train_loader_s1)} (bs={STEP1_BATCH_SIZE})')
print(f'  S2 train batches/epoch: {len(train_loader_s2)} (bs={STEP2_BATCH_SIZE})')


## 6. Model Architecture

### Why rationality embeddings drive the CW prediction

The CheckMate paper frames check-worthiness as a *consequence* of rationality factors:
a claim is check-worthy **because** it contains verifiable information, is harmful,
or needs fact-checker attention. The computational graph:

```
BERT tokens → MulCon attention → g_i (6 label-level embeddings)
                                      │
                    ┌─────────────────┴──────────────────────┐
                    ▼                                        ▼
         R1…R6 classifiers                    Aggregate g_i → CW head
         (multi-label BCE)                    (binary BCE)
```

The CW head sees an **aggregated summary of all 6 rationality embeddings**. With the
fix applied, it now learns the full nuance:
- *"R1+R3 active but non-CW → not enough to trigger check-worthiness"*
- *"R1+R3+R4 active and CW → this combination does trigger check-worthiness"*

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Label-Level Embedding Network  (MulCon Eq. 6)
# ─────────────────────────────────────────────────────────────────────────────
class LabelLevelEmbeddingNetwork(nn.Module):
    """
    g_i = MultiAttBlock(U, r_i, r_i)
    Produces one embedding per rationality label by cross-attending
    learnable class-specific queries U over BERT token representations.
    """
    def __init__(self, embed_dim, num_labels, label_dim, num_heads, dropout=0.1):
        super().__init__()
        self.class_embed = nn.Parameter(torch.randn(num_labels, embed_dim))
        self.q_proj      = nn.Linear(embed_dim, label_dim)
        self.kv_proj     = nn.Linear(embed_dim, label_dim)
        self.cross_attn  = nn.MultiheadAttention(
            embed_dim=label_dim, num_heads=num_heads,
            dropout=dropout, batch_first=True
        )
        self.norm = nn.LayerNorm(label_dim)
        self.ff   = nn.Sequential(
            nn.Linear(label_dim, label_dim), nn.GELU(), nn.Dropout(dropout)
        )

    def forward(self, token_features, attention_mask=None):
        B   = token_features.size(0)
        kv  = self.kv_proj(token_features)
        q   = self.q_proj(self.class_embed).unsqueeze(0).expand(B, -1, -1)
        kpm = (attention_mask == 0) if attention_mask is not None else None
        attn_out, _ = self.cross_attn(q, kv, kv, key_padding_mask=kpm)
        g = self.norm(attn_out + q)
        g = g + self.ff(g)
        return g  # (B, L, label_dim)


# ─────────────────────────────────────────────────────────────────────────────
# Rationality-Aware CW Aggregator
# ─────────────────────────────────────────────────────────────────────────────
class RationalityAggregator(nn.Module):
    """
    Pools the L rationality embeddings into one vector h for the CW classifier.
    mode='mean'     : h = mean over labels
    mode='weighted' : h = softmax-weighted sum (learns which rationality
                          dimensions matter most for CW)
    """
    def __init__(self, label_dim, num_labels, mode='weighted'):
        super().__init__()
        self.mode = mode
        if mode == 'weighted':
            self.attn_w = nn.Linear(label_dim, 1)

    def forward(self, g):
        if self.mode == 'mean':
            return g.mean(dim=1)
        else:
            w = F.softmax(self.attn_w(g), dim=1)
            return (w * g).sum(dim=1)


# ─────────────────────────────────────────────────────────────────────────────
# Projection Head (for contrastive loss)
# ─────────────────────────────────────────────────────────────────────────────
class ProjectionHead(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim)
        )
    def forward(self, x): return self.net(x)


# ─────────────────────────────────────────────────────────────────────────────
# Full Model
# ─────────────────────────────────────────────────────────────────────────────
class CheckMateModel(nn.Module):
    def __init__(
        self,
        bert_name   = BERT_MODEL_NAME,
        num_labels  = NUM_RAT_LABELS,
        embed_dim   = EMBED_DIM,
        label_dim   = LABEL_EMBED_DIM,
        proj_dim    = PROJ_DIM,
        num_heads   = NUM_ATTN_HEADS,
        dropout     = DROPOUT,
        agg_mode    = 'weighted'
    ):
        super().__init__()
        self.num_labels = num_labels
        self.bert            = BertModel.from_pretrained(bert_name)
        self.label_embed_net = LabelLevelEmbeddingNetwork(
            embed_dim=embed_dim, num_labels=num_labels,
            label_dim=label_dim, num_heads=num_heads, dropout=dropout
        )
        self.rat_classifiers = nn.ModuleList([
            nn.Sequential(nn.Dropout(dropout), nn.Linear(label_dim, 1))
            for _ in range(num_labels)
        ])
        self.aggregator = RationalityAggregator(label_dim, num_labels, mode=agg_mode)
        self.cw_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(label_dim, label_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(label_dim // 2, 1)
        )
        self.projector = ProjectionHead(label_dim, label_dim, proj_dim, dropout)

    def forward(self, input_ids, attention_mask, return_proj=False):
        token_features = self.bert(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state
        g = self.label_embed_net(token_features, attention_mask)
        rat_logits = torch.cat(
            [self.rat_classifiers[j](g[:, j, :]) for j in range(self.num_labels)],
            dim=-1
        )
        h        = self.aggregator(g)
        cw_logit = self.cw_head(h)
        if return_proj:
            z = F.normalize(self.projector(g), dim=-1)
            return cw_logit, rat_logits, z
        return cw_logit, rat_logits


model = CheckMateModel().to(DEVICE)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: total={total:,} | trainable={trainable:,}')

## 7. Loss Functions

**Fixes applied:**

1. **`compute_loss` – rationality BCE**: already trained on all rows → unchanged, but now
   uses *true* labels for non-CW rows (since we no longer zero them in `load_data`).

2. **`compute_loss` – contrastive loss**: changed from `cw_mask` (CW rows only) to
   `active_mask` (any row with ≥1 active rationality label). This lets non-CW rows
   that have genuine rationality signals participate as negative pairs, making the
   contrastive loss richer and more grounded.

In [ ]:
cw_bce_fn  = nn.BCEWithLogitsLoss()
rat_bce_fn = nn.BCEWithLogitsLoss()


class MultiLabelSupConLoss(nn.Module):
    """
    MulCon Supervised Contrastive Loss (Eq. 8).
    Applied to the rationality label-level projections z.
    Anchor (i,j): embedding of sample i under active label j.
    Positive P(i,j): embeddings of other samples with the same active label j.
    """
    def __init__(self, temperature=TEMPERATURE):
        super().__init__()
        self.tau = temperature

    def forward(self, z, rat_labels):
        """
        z          : (B, L, proj_dim) L2-normalised
        rat_labels : (B, L) binary
        """
        B, L, D = z.shape
        dev     = z.device
        active  = rat_labels.bool()
        z_flat  = z[active]           # (N_active, D)
        lbl_ids = torch.where(active)[1]
        N = z_flat.size(0)
        if N < 2:
            return torch.tensor(0.0, device=dev, requires_grad=True)

        sim      = torch.matmul(z_flat, z_flat.T) / self.tau
        same_lbl = (lbl_ids.unsqueeze(0) == lbl_ids.unsqueeze(1)).float()
        self_mask = torch.eye(N, device=dev)
        pos_mask  = same_lbl * (1 - self_mask)

        has_pos = pos_mask.sum(dim=1) > 0
        if not has_pos.any():
            return torch.tensor(0.0, device=dev, requires_grad=True)

        sim_m     = sim - self_mask * 1e9
        log_denom = torch.logsumexp(sim_m, dim=1, keepdim=True)
        log_prob  = sim - log_denom
        num_pos   = pos_mask.sum(dim=1).clamp(min=1)
        anchor_loss = -(pos_mask * log_prob).sum(dim=1) / num_pos
        return anchor_loss[has_pos].mean()


con_loss_fn = MultiLabelSupConLoss()


def compute_loss(cw_logit, rat_logits, z, cw_label, rat_labels,
                 use_contrastive=False):
    """
    L = λ_cw * L_CW  +  λ_rat * L_BCE(R1–R6)  [+  γ * L_con]

    Rationality BCE:
      Applied to ALL samples using their TRUE labels (including non-CW rows).
      This teaches the model the full picture:
        - CW=1, R1=1, R3=1 → check-worthy with those rationality signals active
        - CW=0, R1=1, R3=0 → not check-worthy even though R1 is active
      Previously, non-CW rows were zeroed before reaching here, so this
      distinction could never be learned.

    Contrastive loss:
      FIX: Uses any row with ≥1 active rationality label (not just CW rows).
      Non-CW rows with genuine rationality signals now contribute as negative
      pairs, giving the contrastive loss richer and more accurate supervision.
    """
    l_cw  = cw_bce_fn(cw_logit.squeeze(-1), cw_label)
    l_rat = rat_bce_fn(rat_logits, rat_labels)
    total = LAMBDA_CW * l_cw + LAMBDA_RAT * l_rat

    l_con = torch.tensor(0.0, device=cw_label.device)
    if use_contrastive and z is not None:
        # FIX: use any row with ≥1 active rationality label, not just CW rows
        active_mask = rat_labels.sum(dim=1) > 0
        if active_mask.sum() >= 2:
            l_con = con_loss_fn(z[active_mask], rat_labels[active_mask])
            total = total + GAMMA * l_con

    return total, l_cw, l_rat, l_con


print('Loss functions ready.')

## 8. Evaluation

**Fix**: Rationality metrics are now computed on all rows with ≥1 active rationality
label, not just CW rows. This reflects the corrected training setup where non-CW rows
can also have active rationality labels.

In [ ]:
def evaluate(model, loader, device=DEVICE, threshold=THRESHOLD):
    model.eval()
    all_cw_pred, all_cw_true   = [], []
    all_rat_pred, all_rat_true = [], []
    total_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            cw   = batch['cw_label'].to(device)
            rat  = batch['rat_labels'].to(device)

            cw_logit, rat_logits = model(ids, mask)
            loss, *_ = compute_loss(cw_logit, rat_logits, None, cw, rat)
            total_loss += loss.item()

            all_cw_pred.append((torch.sigmoid(cw_logit.squeeze(-1)) >= threshold).cpu().numpy())
            all_cw_true.append(cw.cpu().numpy())
            all_rat_pred.append((torch.sigmoid(rat_logits) >= threshold).cpu().numpy())
            all_rat_true.append(rat.cpu().numpy())

    cw_pred  = np.concatenate(all_cw_pred)
    cw_true  = np.concatenate(all_cw_true)
    rat_pred = np.vstack(all_rat_pred)
    rat_true = np.vstack(all_rat_true)

    # ── CW metrics ─────────────────────────────────────────────────────────
    cw_acc      = accuracy_score(cw_true, cw_pred)
    cw_macro_f1 = f1_score(cw_true, cw_pred, average='macro',   zero_division=0)
    cw_micro_f1 = f1_score(cw_true, cw_pred, average='micro',   zero_division=0)
    cw_f1s      = f1_score(cw_true, cw_pred, average=None,      zero_division=0)
    cw_precision = precision_score(cw_true, cw_pred, average='macro', zero_division=0)
    cw_recall    = recall_score(cw_true, cw_pred, average='macro',    zero_division=0)
    try:
        cw_auc = roc_auc_score(cw_true, cw_pred)
    except Exception:
        cw_auc = float('nan')

    cw_f1_ncw = float(cw_f1s[0]) if len(cw_f1s) > 0 else 0.0
    cw_f1_cw  = float(cw_f1s[1]) if len(cw_f1s) > 1 else 0.0

    # ── Rationality metrics ─────────────────────────────────────────────────
    # FIX: evaluate on ALL rows with ≥1 active rationality label
    # (previously only CW rows — now correctly includes non-CW rows with
    # genuine rationality labels)
    active_idx = rat_true.sum(axis=1) > 0
    if active_idx.sum() > 0:
        rat_macro_f1  = f1_score(rat_true[active_idx], rat_pred[active_idx],
                                 average='macro',   zero_division=0)
        rat_micro_f1  = f1_score(rat_true[active_idx], rat_pred[active_idx],
                                 average='micro',   zero_division=0)
        rat_per_label = f1_score(rat_true[active_idx], rat_pred[active_idx],
                                 average=None,      zero_division=0)
        rat_precision = precision_score(rat_true[active_idx], rat_pred[active_idx],
                                        average='macro',      zero_division=0)
        rat_recall    = recall_score(rat_true[active_idx], rat_pred[active_idx],
                                     average='macro',       zero_division=0)
        rat_prec_per  = precision_score(rat_true[active_idx], rat_pred[active_idx],
                                        average=None,         zero_division=0)
        rat_rec_per   = recall_score(rat_true[active_idx], rat_pred[active_idx],
                                     average=None,          zero_division=0)
    else:
        rat_macro_f1 = rat_micro_f1 = rat_precision = rat_recall = 0.0
        rat_per_label = np.zeros(NUM_RAT_LABELS)
        rat_prec_per  = np.zeros(NUM_RAT_LABELS)
        rat_rec_per   = np.zeros(NUM_RAT_LABELS)

    return {
        'loss':          total_loss / len(loader),
        'cw_acc':        cw_acc,
        'cw_macro_f1':   cw_macro_f1,
        'cw_micro_f1':   cw_micro_f1,
        'cw_f1_cw':      cw_f1_cw,
        'cw_f1_ncw':     cw_f1_ncw,
        'cw_precision':  cw_precision,
        'cw_recall':     cw_recall,
        'cw_auc':        cw_auc,
        'rat_macro_f1':  rat_macro_f1,
        'rat_micro_f1':  rat_micro_f1,
        'rat_precision': rat_precision,
        'rat_recall':    rat_recall,
        'rat_per_label': rat_per_label,
        'rat_prec_per':  rat_prec_per,
        'rat_rec_per':   rat_rec_per,
    }


def print_metrics(split, m):
    W = 65
    print('\n' + '='*W)
    print(f'  {split} METRICS')
    print('='*W)
    print(f'  Loss                    : {m["loss"]:.4f}')
    print(f'\n  ── CHECK-WORTHINESS ─────────────────────────────────')
    print(f'  Accuracy                : {m["cw_acc"]:.4f}')
    print(f'  Macro F1                : {m["cw_macro_f1"]:.4f}')
    print(f'  Micro F1                : {m["cw_micro_f1"]:.4f}')
    print(f'  Macro Precision         : {m["cw_precision"]:.4f}')
    print(f'  Macro Recall            : {m["cw_recall"]:.4f}')
    print(f'  AUC-ROC                 : {m["cw_auc"]:.4f}')
    print(f'  F1 (Check-Worthy)       : {m["cw_f1_cw"]:.4f}')
    print(f'  F1 (Non-Check-Worthy)   : {m["cw_f1_ncw"]:.4f}')
    print(f'\n  ── RATIONALITY (rows with ≥1 active label) ───────────')
    print(f'  Macro F1                : {m["rat_macro_f1"]:.4f}')
    print(f'  Micro F1                : {m["rat_micro_f1"]:.4f}')
    print(f'  Macro Precision         : {m["rat_precision"]:.4f}')
    print(f'  Macro Recall            : {m["rat_recall"]:.4f}')
    print(f'\n  {"Label":<35} {"F1":>6}  {"Prec":>6}  {"Rec":>6}')
    print(f'  {"-"*57}')
    for lbl, f1, pr, rc in zip(LABEL_COLS, m['rat_per_label'],
                                m['rat_prec_per'], m['rat_rec_per']):
        print(f'  {lbl:<35} {f1:>6.4f}  {pr:>6.4f}  {rc:>6.4f}')
    print('='*W)


print('Evaluation utilities ready.')

## 9. Step 1 – Pre-training (CW + Rationality BCE)

**What it does**: Jointly trains the BERT encoder + label-level embedding network +
rationality classifiers + CW head using standard BCE losses only.

**Key design choices**:
- Separate LR groups: BERT at `STEP1_LR`, non-BERT heads at `STEP1_LR × 10`
  (BERT already has learned representations; non-BERT heads start random)
- High weight decay on BERT (`0.05`) to reduce train/val loss divergence
- Early stopping with `PATIENCE=3` — saves best checkpoint and stops when plateau
- No contrastive loss in Step 1 (embeddings not yet discriminative enough)


In [ ]:
def train_step1(model, train_loader, val_loader,
                epochs=STEP1_EPOCHS, lr=STEP1_LR, patience=PATIENCE):
    print('='*65)
    print('STEP 1: Pre-training  (L_CW + L_BCE, no contrastive loss)')
    print('='*65)

    # Separate LR for BERT vs non-BERT parameters
    # BERT: fine-tune gently with high weight decay (already learned)
    # Non-BERT: learn faster, lower weight decay (randomly initialised)
    bert_params  = list(model.bert.parameters())
    other_params = [p for n, p in model.named_parameters() if 'bert' not in n]

    opt = AdamW([
        {'params': bert_params,  'lr': lr,       'weight_decay': 0.05},
        {'params': other_params, 'lr': lr * 10,  'weight_decay': WEIGHT_DECAY},
    ])
    sch = OneCycleLR(opt, max_lr=[lr, lr * 10],
                     steps_per_epoch=len(train_loader),
                     epochs=epochs, pct_start=0.1)

    best_f1, ckpt    = 0.0, os.path.join(OUTPUT_DIR, 'step1_best.pt')
    patience_counter = 0
    history = {'train_loss':[], 'val_loss':[], 'val_cw_f1':[], 'val_rat_f1':[]}

    for epoch in range(1, epochs + 1):
        model.train()
        ep_loss = 0.0
        for batch in train_loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            cw   = batch['cw_label'].to(DEVICE)
            rat  = batch['rat_labels'].to(DEVICE)
            opt.zero_grad()
            cw_logit, rat_logits = model(ids, mask)
            loss, *_ = compute_loss(cw_logit, rat_logits, None, cw, rat)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step()
            ep_loss += loss.item()

        m = evaluate(model, val_loader)
        history['train_loss'].append(ep_loss / len(train_loader))
        history['val_loss'].append(m['loss'])
        history['val_cw_f1'].append(m['cw_macro_f1'])
        history['val_rat_f1'].append(m['rat_macro_f1'])

        improved = m['cw_macro_f1'] > best_f1
        marker   = '  ✓ Saved' if improved else f'  patience {patience_counter+1}/{patience}'
        print(f'Ep {epoch:>3}/{epochs} | '
              f'Train: {ep_loss/len(train_loader):.4f} | '
              f'Val: {m["loss"]:.4f} | '
              f'CW-F1: {m["cw_macro_f1"]:.4f} | '
              f'RAT-F1: {m["rat_macro_f1"]:.4f}{marker}')

        if improved:
            best_f1          = m['cw_macro_f1']
            patience_counter = 0
            torch.save(model.state_dict(), ckpt)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'  Early stopping at epoch {epoch}')
                break

    print(f'\nStep 1 done. Best Val CW Macro F1: {best_f1:.4f}')
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return history


history_s1 = train_step1(model, train_loader_s1, val_loader)


## 10. Step 2 – Contrastive Fine-tuning

**What it does**: Fine-tunes only the non-BERT components (label embedding network,
aggregator, classifiers, projector) using BCE + supervised contrastive loss.

**Key design choices**:
- **BERT is frozen in Step 2** — the most important fix. Running SGD on a BERT
  that was already well-trained by AdamW in Step 1 degrades the representations
  and causes the full model to score *worse* than the "w/o Contrastive" ablation.
  Freezing BERT means only the MulCon-specific layers are updated.
- Larger batch size (`STEP2_BATCH_SIZE=32`) gives more active label pairs per batch,
  which is essential for the contrastive loss to fire meaningfully.
- `STEP2_LR=5e-5` is calibrated for SGD on small non-BERT layers.
- Early stopping again with `PATIENCE=3`.


In [ ]:
def train_step2(model, train_loader, val_loader,
                epochs=STEP2_EPOCHS, lr=STEP2_LR, patience=PATIENCE):
    print('='*65)
    print('STEP 2: Contrastive fine-tuning  (L_CW + L_BCE + gamma*L_con)')
    print('='*65)

    # FREEZE BERT — critical fix.
    # Step 2 should only update the MulCon-specific layers.
    # Running SGD on BERT after AdamW fine-tuning degrades the representations
    # and causes the full model to underperform the "w/o Contrastive" ablation.
    for param in model.bert.parameters():
        param.requires_grad = False

    trainable = [p for n, p in model.named_parameters()
                 if 'bert' not in n and p.requires_grad]
    n_trainable = sum(p.numel() for p in trainable)
    print(f'  BERT frozen. Trainable params: {n_trainable:,}')
    print(f'  (label_embed_net, aggregator, rat_classifiers, cw_head, projector)')

    opt = torch.optim.SGD(trainable, lr=lr, momentum=0.9, weight_decay=WEIGHT_DECAY)
    sch = StepLR(opt, step_size=max(1, epochs // 2), gamma=0.1)

    best_f1, ckpt    = 0.0, os.path.join(OUTPUT_DIR, 'step2_best.pt')
    patience_counter = 0
    history = {
        'train_loss':[], 'train_cw':[], 'train_rat':[], 'train_con':[],
        'val_loss':[], 'val_cw_f1':[], 'val_rat_f1':[]
    }

    for epoch in range(1, epochs + 1):
        model.train()
        e = {'loss': 0, 'cw': 0, 'rat': 0, 'con': 0}

        for batch in train_loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            cw   = batch['cw_label'].to(DEVICE)
            rat  = batch['rat_labels'].to(DEVICE)
            opt.zero_grad()
            cw_logit, rat_logits, z = model(ids, mask, return_proj=True)
            loss, l_cw, l_rat, l_con = compute_loss(
                cw_logit, rat_logits, z, cw, rat, use_contrastive=True)
            loss.backward()
            nn.utils.clip_grad_norm_(trainable, 1.0)
            opt.step()
            e['loss'] += loss.item()
            e['cw']   += l_cw.item()
            e['rat']  += l_rat.item()
            e['con']  += l_con.item()

        sch.step()
        n = len(train_loader)
        for k in e:
            history[f'train_{k}'].append(e[k] / n)

        m = evaluate(model, val_loader)
        history['val_loss'].append(m['loss'])
        history['val_cw_f1'].append(m['cw_macro_f1'])
        history['val_rat_f1'].append(m['rat_macro_f1'])

        improved = m['cw_macro_f1'] > best_f1
        marker   = '  ✓ Saved' if improved else f'  patience {patience_counter+1}/{patience}'
        print(f'Ep {epoch:>3}/{epochs} | '
              f'Loss: {e["loss"]/n:.4f} '
              f'(CW={e["cw"]/n:.4f} RAT={e["rat"]/n:.4f} CON={e["con"]/n:.4f}) | '
              f'CW-F1: {m["cw_macro_f1"]:.4f} | RAT-F1: {m["rat_macro_f1"]:.4f}{marker}')

        if improved:
            best_f1          = m['cw_macro_f1']
            patience_counter = 0
            torch.save(model.state_dict(), ckpt)
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'  Early stopping at epoch {epoch}')
                break

    # Unfreeze BERT for inference
    for param in model.bert.parameters():
        param.requires_grad = True

    print(f'\nStep 2 done. Best Val CW Macro F1: {best_f1:.4f}')
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return history


history_s2 = train_step2(model, train_loader_s2, val_loader)


## 11. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))

axes[0,0].plot(history_s1['train_loss'], label='Train', color='steelblue')
axes[0,0].plot(history_s1['val_loss'],   label='Val',   color='tomato')
axes[0,0].set_title('Step 1 – Total Loss'); axes[0,0].legend(); axes[0,0].grid(alpha=.3)

axes[0,1].plot(history_s1['val_cw_f1'],  label='CW Macro F1',  color='steelblue', marker='o')
axes[0,1].plot(history_s1['val_rat_f1'], label='RAT Macro F1', color='darkorange', marker='s')
axes[0,1].set_title('Step 1 – Validation F1'); axes[0,1].legend(); axes[0,1].grid(alpha=.3)

axes[0,2].axis('off')

axes[1,0].plot(history_s2['train_cw'],   label='L_CW',   color='steelblue')
axes[1,0].plot(history_s2['train_rat'],  label='L_RAT',  color='seagreen')
axes[1,0].plot(history_s2['train_con'],  label='L_CON',  color='darkorange')
axes[1,0].plot(history_s2['train_loss'], label='Total',  color='purple', linestyle='--')
axes[1,0].set_title('Step 2 – Loss Breakdown'); axes[1,0].legend(); axes[1,0].grid(alpha=.3)

axes[1,1].plot(history_s2['val_cw_f1'],  label='CW Macro F1',  color='steelblue', marker='o')
axes[1,1].plot(history_s2['val_rat_f1'], label='RAT Macro F1', color='darkorange', marker='s')
axes[1,1].set_title('Step 2 – Validation F1'); axes[1,1].legend(); axes[1,1].grid(alpha=.3)

all_cw = history_s1['val_cw_f1'] + history_s2['val_cw_f1']
axes[1,2].plot(all_cw, color='steelblue', marker='o', markersize=4)
axes[1,2].axvline(STEP1_EPOCHS-0.5, color='red', linestyle='--', label='Step1→Step2')
axes[1,2].set_title('CW Macro F1 – All Epochs'); axes[1,2].legend(); axes[1,2].grid(alpha=.3)

for ax in axes.flat: ax.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 12. Test Set Evaluation

In [ ]:
test_m = evaluate(model, test_loader)
print_metrics('TEST', test_m)

# Full classification reports
model.eval()
all_cw_pred, all_cw_true   = [], []
all_rat_pred, all_rat_true = [], []

with torch.no_grad():
    for batch in test_loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        cw_logit, rat_logits = model(ids, mask)
        all_cw_pred.append((torch.sigmoid(cw_logit.squeeze(-1).cpu()) >= THRESHOLD).numpy())
        all_rat_pred.append((torch.sigmoid(rat_logits.cpu())           >= THRESHOLD).numpy())
        all_cw_true.append(batch['cw_label'].numpy())
        all_rat_true.append(batch['rat_labels'].numpy())

cw_pred  = np.concatenate(all_cw_pred)
cw_true  = np.concatenate(all_cw_true)
rat_pred = np.vstack(all_rat_pred)
rat_true = np.vstack(all_rat_true)

print('\n' + '='*55)
print('CHECK-WORTHINESS REPORT')
print('='*55)
print(classification_report(cw_true, cw_pred,
      target_names=['Non-CW','Check-Worthy'], zero_division=0))

print('\n' + '='*55)
print('RATIONALITY LABELS (rows with ≥1 active label)')
print('='*55)
active_idx = rat_true.sum(axis=1) > 0
for i, col in enumerate(LABEL_COLS):
    print(f'\n── {col} ──')
    print(classification_report(rat_true[active_idx, i], rat_pred[active_idx, i],
          target_names=['No','Yes'], zero_division=0))

## 13. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cm = confusion_matrix(cw_true, cw_pred)
ax = axes[0]
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Non-CW','CW'], fontsize=12)
ax.set_yticklabels(['Non-CW','CW'], fontsize=12)
ax.set_xlabel('Predicted', fontsize=12); ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Check-Worthiness Confusion Matrix', fontsize=13, fontweight='bold')
for r in range(2):
    for c in range(2):
        ax.text(c, r, str(cm[r,c]), ha='center', va='center', fontsize=16,
                color='white' if cm[r,c] > cm.max()/2 else 'black')
plt.colorbar(im, ax=ax)

ax = axes[1]
colors = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B3','#937860']
bars = ax.bar(LABEL_COLS, test_m['rat_per_label'], color=colors, edgecolor='white', linewidth=1.2)
for bar, v in zip(bars, test_m['rat_per_label']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{v:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.axhline(test_m['rat_macro_f1'], color='black', linestyle='--',
           label=f'Macro F1={test_m["rat_macro_f1"]:.3f}')
ax.set_title('Rationality Label F1 (rows with ≥1 active label)', fontsize=13, fontweight='bold')
ax.set_ylabel('F1 Score'); ax.set_ylim(0, 1.05)
ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'test_results.png'), dpi=150, bbox_inches='tight')
plt.show()

## 14. Ablation Study

Five conditions, each isolating exactly one component of MulCon.
All retrained conditions use the **same two-phase training** as the full model
(AdamW Phase 1 + SGD Phase 2 with BERT frozen + early stopping) — ensuring fair comparison.

| Condition | What is removed | What it isolates |
|-----------|----------------|-----------------|
| **Full Model** | Nothing | Upper bound |
| **w/o Contrastive** | L_con in Phase 2 | Value of contrastive fine-tuning |
| **w/o Label Attention** | Cross-attention → mean-pool | Value of per-label cross-attention (MulCon Eq. 6) |
| **w/o Weighted Agg** | Learned agg → mean agg | Value of learned aggregation over label dims |
| **w/o Multi-task** | L_rat zeroed out | Value of joint rationality supervision |

**Note**: "w/o Rationality Grounding" from earlier versions has been removed —
it used an untrained projection layer which made it an invalid ablation.


In [ ]:
import os
from copy import deepcopy


def ablation_two_phase(model_obj, name,
                       s1_epochs=STEP1_EPOCHS, s1_lr=STEP1_LR,
                       s2_epochs=STEP2_EPOCHS, s2_lr=STEP2_LR,
                       use_contrastive_s2=True,
                       loss_fn=None):
    """
    Replicates the full two-phase training schedule for ablation conditions.

    Phase 1: AdamW + OneCycleLR + early stopping (BCE only)
             Separate LR: BERT at s1_lr, non-BERT at s1_lr*10
    Phase 2: SGD + StepLR + early stopping (BCE [+ contrastive])
             BERT frozen — only MulCon-specific layers updated

    loss_fn: optional loss override for multi-task ablation.
    """
    _loss     = loss_fn if loss_fn is not None else compute_loss
    model_obj = model_obj.to(DEVICE)

    # ── Phase 1 ─────────────────────────────────────────────────────────────
    print(f'  [{name}] Phase 1 ({s1_epochs} epochs max, patience={PATIENCE})...')
    loader_s1 = make_loader(train_ds, STEP1_BATCH_SIZE, shuffle=True)

    bert_p  = list(model_obj.bert.parameters())
    other_p = [p for n, p in model_obj.named_parameters() if 'bert' not in n]
    opt1 = AdamW([
        {'params': bert_p,  'lr': s1_lr,      'weight_decay': 0.05},
        {'params': other_p, 'lr': s1_lr * 10, 'weight_decay': WEIGHT_DECAY},
    ])
    sch1 = OneCycleLR(opt1, max_lr=[s1_lr, s1_lr * 10],
                      steps_per_epoch=len(loader_s1),
                      epochs=s1_epochs, pct_start=0.1)

    best_f1, best_state, patience_ctr = 0.0, None, 0

    for epoch in range(1, s1_epochs + 1):
        model_obj.train()
        ep_loss = 0.0
        for batch in loader_s1:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            cw   = batch['cw_label'].to(DEVICE)
            rat  = batch['rat_labels'].to(DEVICE)
            opt1.zero_grad()
            cw_logit, rat_logits = model_obj(ids, mask)
            loss, *_ = _loss(cw_logit, rat_logits, None, cw, rat, use_contrastive=False)
            loss.backward()
            nn.utils.clip_grad_norm_(model_obj.parameters(), 1.0)
            opt1.step(); sch1.step()
            ep_loss += loss.item()

        m = evaluate(model_obj, val_loader)
        if epoch % max(1, s1_epochs // 4) == 0 or epoch == s1_epochs:
            print(f'    Ep {epoch:>2}/{s1_epochs} | '
                  f'Loss: {ep_loss/len(loader_s1):.4f} | '
                  f'CW-F1: {m["cw_macro_f1"]:.4f}')

        if m['cw_macro_f1'] > best_f1:
            best_f1     = m['cw_macro_f1']
            best_state  = {k: v.cpu().clone() for k, v in model_obj.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f'    Early stopping at epoch {epoch}')
                break

    model_obj.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    print(f'  [{name}] Phase 1 done. Best CW-F1: {best_f1:.4f}')

    # ── Phase 2 (BERT frozen) ───────────────────────────────────────────────
    print(f'  [{name}] Phase 2 ({s2_epochs} epochs, contrastive={use_contrastive_s2}, BERT frozen)...')
    for param in model_obj.bert.parameters():
        param.requires_grad = False

    loader_s2  = make_loader(train_ds, STEP2_BATCH_SIZE, shuffle=True)
    trainable  = [p for n, p in model_obj.named_parameters()
                  if 'bert' not in n and p.requires_grad]
    opt2 = torch.optim.SGD(trainable, lr=s2_lr, momentum=0.9, weight_decay=WEIGHT_DECAY)
    sch2 = StepLR(opt2, step_size=max(1, s2_epochs // 2), gamma=0.1)

    best_f1, best_state, patience_ctr = 0.0, None, 0

    for epoch in range(1, s2_epochs + 1):
        model_obj.train()
        ep_loss = 0.0
        for batch in loader_s2:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            cw   = batch['cw_label'].to(DEVICE)
            rat  = batch['rat_labels'].to(DEVICE)
            opt2.zero_grad()
            if use_contrastive_s2:
                cw_logit, rat_logits, z = model_obj(ids, mask, return_proj=True)
                loss, *_ = _loss(cw_logit, rat_logits, z, cw, rat, use_contrastive=True)
            else:
                cw_logit, rat_logits = model_obj(ids, mask)
                loss, *_ = _loss(cw_logit, rat_logits, None, cw, rat, use_contrastive=False)
            loss.backward()
            nn.utils.clip_grad_norm_(trainable, 1.0)
            opt2.step()
            ep_loss += loss.item()

        sch2.step()
        m = evaluate(model_obj, val_loader)

        if m['cw_macro_f1'] > best_f1:
            best_f1     = m['cw_macro_f1']
            best_state  = {k: v.cpu().clone() for k, v in model_obj.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f'    Early stopping at epoch {epoch}')
                break

    for param in model_obj.bert.parameters():
        param.requires_grad = True

    if best_state is not None:
        model_obj.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    print(f'  [{name}] Phase 2 done. Best CW-F1: {best_f1:.4f}')
    return model_obj


# ─────────────────────────────────────────────────────────────────────────────
# CONDITION 1: Full Model  (load from step2_best checkpoint)
# Verify checkpoint quality before comparing ablations.
# If val F1 is unexpectedly low, rerun training before running ablations.
# ─────────────────────────────────────────────────────────────────────────────
print('='*70)
print('ABLATION STUDY')
print('='*70)

full_model = CheckMateModel().to(DEVICE)
full_model.load_state_dict(
    torch.load(os.path.join(OUTPUT_DIR, 'step2_best.pt'), map_location=DEVICE))

m_full_val = evaluate(full_model, val_loader)
print(f'\n[Checkpoint check] Val CW Macro F1: {m_full_val["cw_macro_f1"]:.4f}')
m_full = evaluate(full_model, test_loader)
print(f'[1/5] Full Model — Test CW Macro F1: {m_full["cw_macro_f1"]:.4f}')


# ─────────────────────────────────────────────────────────────────────────────
# CONDITION 2: w/o Contrastive Loss
# Same architecture + two-phase training, but Phase 2 uses BCE only (no L_con).
# With BERT frozen in Phase 2, this is now a valid isolated comparison.
# ─────────────────────────────────────────────────────────────────────────────
print('\n[2/5] w/o Contrastive Loss...')
model_no_con = CheckMateModel().to(DEVICE)
model_no_con = ablation_two_phase(model_no_con, 'w/o Contrastive',
                                   use_contrastive_s2=False)
m_no_con = evaluate(model_no_con, test_loader)


# ─────────────────────────────────────────────────────────────────────────────
# CONDITION 3: w/o Label-Level Attention
# Replace per-label cross-attention with masked mean-pool of BERT tokens.
# All 6 label embeddings become identical — no label specialisation.
# ─────────────────────────────────────────────────────────────────────────────
print('\n[3/5] w/o Label-Level Attention...')

class MeanPoolLabelEmbedding(nn.Module):
    """Replaces LabelLevelEmbeddingNetwork with mean-pool of BERT tokens."""
    def __init__(self, embed_dim, num_labels, label_dim, **kwargs):
        super().__init__()
        self.proj       = nn.Linear(embed_dim, label_dim)
        self.norm       = nn.LayerNorm(label_dim)
        self.num_labels = num_labels

    def forward(self, token_features, attention_mask=None):
        if attention_mask is not None:
            mask   = attention_mask.unsqueeze(-1).float()
            pooled = (token_features * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        else:
            pooled = token_features.mean(1)
        h = self.norm(self.proj(pooled))
        return h.unsqueeze(1).expand(-1, self.num_labels, -1).contiguous()

class CheckMateNoLabelAttn(CheckMateModel):
    def __init__(self):
        super().__init__()
        self.label_embed_net = MeanPoolLabelEmbedding(
            EMBED_DIM, NUM_RAT_LABELS, LABEL_EMBED_DIM)

model_no_attn = CheckMateNoLabelAttn().to(DEVICE)
model_no_attn = ablation_two_phase(model_no_attn, 'w/o Label Attention',
                                    use_contrastive_s2=True)
m_no_attn = evaluate(model_no_attn, test_loader)


# ─────────────────────────────────────────────────────────────────────────────
# CONDITION 4: w/o Weighted Aggregation
# Simple mean aggregation instead of learned softmax attention over labels.
# ─────────────────────────────────────────────────────────────────────────────
print('\n[4/5] w/o Weighted Aggregation...')

class CheckMateMeanAgg(CheckMateModel):
    def __init__(self): super().__init__(agg_mode='mean')

model_mean_agg = CheckMateMeanAgg().to(DEVICE)
model_mean_agg = ablation_two_phase(model_mean_agg, 'w/o Weighted Agg',
                                     use_contrastive_s2=True)
m_mean_agg = evaluate(model_mean_agg, test_loader)


# ─────────────────────────────────────────────────────────────────────────────
# CONDITION 5: w/o Multi-task Learning
# L_rat zeroed out — only CW BCE drives the model.
# Contrastive also disabled: SCL needs meaningful rationality embeddings
# which only emerge when L_rat provides gradient signal.
# Uses clean loss_fn override — no global function swap.
# ─────────────────────────────────────────────────────────────────────────────
print('\n[5/5] w/o Multi-task Learning...')

def compute_loss_cw_only(cw_logit, rat_logits, z, cw_label, rat_labels,
                          use_contrastive=False):
    """CW BCE only. L_rat and L_con are zeroed out."""
    l_cw = cw_bce_fn(cw_logit.squeeze(-1), cw_label)
    l_rat = torch.zeros(1, device=cw_label.device)
    l_con = torch.zeros(1, device=cw_label.device)
    return l_cw, l_cw, l_rat, l_con

model_cw_only = CheckMateModel().to(DEVICE)
model_cw_only = ablation_two_phase(model_cw_only, 'w/o Multi-task',
                                    use_contrastive_s2=False,
                                    loss_fn=compute_loss_cw_only)
m_cw_only = evaluate(model_cw_only, test_loader)


# ─────────────────────────────────────────────────────────────────────────────
# RESULTS TABLE
# ─────────────────────────────────────────────────────────────────────────────
ablation_rows = [
    ('Full MulCon  (all components)',             m_full),
    ('w/o Contrastive Loss        (-L_con)',      m_no_con),
    ('w/o Label-Level Attention   (-cross-attn)', m_no_attn),
    ('w/o Weighted Aggregation    (-attn-pool)',  m_mean_agg),
    ('w/o Multi-task Learning     (-L_rat)',      m_cw_only),
]

SEP = '=' * 110
print('\n\n' + SEP)
print('  ABLATION STUDY — TEST SET RESULTS')
print(SEP)
hdr = (f'  {"Condition":<46} {"CW-Acc":>7} {"CW-MacF1":>9} '
       f'{"CW-F1(+)":>9} {"CW-F1(-)":>9} {"RAT-MacF1":>10}')
print(hdr)
print('  ' + '-' * 96)

ref = m_full
for name, m in ablation_rows:
    is_full = 'Full MulCon' in name
    def v(key, w=9): return f'{m[key]:.4f}'.rjust(w)
    if is_full:
        delta_cw, delta_rat = '', ''
    else:
        dcw  = m['cw_macro_f1']  - ref['cw_macro_f1']
        drat = m['rat_macro_f1'] - ref['rat_macro_f1']
        delta_cw  = f'({dcw:+.4f})'
        delta_rat = f'({drat:+.4f})'
    print(f'  {name:<46} {v("cw_acc",7)} {v("cw_macro_f1")} '
          f'{v("cw_f1_cw")} {v("cw_f1_ncw")} {v("rat_macro_f1",10)}'
          f'  {delta_cw:<11} {delta_rat}')

print(SEP)

# Per-label breakdown
short_labels = ['Verifiable', 'FalseInfo', 'PublicInt', 'Harmful', 'FactCheck', 'GovtAttn']
print('\n  PER-LABEL RATIONALITY F1  (Delta vs Full Model):')
print('  ' + '-' * 100)
W = 11
print(f'  {"Condition":<46}' + ''.join(f' {lbl[:W]:>{W}}' for lbl in short_labels))
print('  ' + '-' * 100)
for name, m in ablation_rows:
    is_full = 'Full MulCon' in name
    row = f'  {name:<46}'
    for f1, rf1 in zip(m['rat_per_label'], ref['rat_per_label']):
        if is_full:
            row += f' {f1:{W}.4f}'
        else:
            d = f1 - rf1
            mk = 'v' if d < -0.005 else ('^' if d > 0.005 else ' ')
            row += f' {f"{f1:.3f}{mk}{abs(d):.2f}":>{W}}'
    print(row)
print(SEP)

print('\n  What each condition isolates:')
print('  Full vs w/o Contrastive     -> value of MulCon contrastive fine-tuning (L_con)')
print('  Full vs w/o Label Attention -> value of per-label cross-attention (MulCon Eq. 6)')
print('  Full vs w/o Weighted Agg    -> value of learned aggregation over rationality dims')
print('  Full vs w/o Multi-task      -> value of joint rationality supervision on CW')


## 15. Inference – Joint Prediction with Explanation

In [ ]:
def predict(texts, model, tokenizer,
            threshold=THRESHOLD, device=DEVICE, batch_size=32):
    """
    Batched inference. Returns a DataFrame with per-claim:
      - check_worthy (0/1) + cw_prob
      - R1…R6 predictions + probabilities
      - Human-readable explanation grounded in active rationality labels
    """
    RAT_DESCRIPTIONS = {col: desc for col, desc in zip(LABEL_COLS, [
        'contains verifiable factual information',
        'contains potentially false or misleading information',
        'is of general public interest',
        'could be harmful to individuals or society',
        'requires fact-checker attention',
        'requires government attention',
    ])}

    model.eval()
    model.to(device)
    all_cw_prob, all_rat_prob = [], []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start : start + batch_size]
        enc = tokenizer(
            batch_texts, max_length=MAX_SEQ_LEN,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        ids  = enc['input_ids'].to(device)
        mask = enc['attention_mask'].to(device)
        with torch.no_grad():
            cw_logit, rat_logits = model(ids, mask)
            all_cw_prob.append(torch.sigmoid(cw_logit.squeeze(-1)).cpu())
            all_rat_prob.append(torch.sigmoid(rat_logits).cpu())

    cw_prob  = torch.cat(all_cw_prob).numpy()
    rat_prob = torch.cat(all_rat_prob).numpy()
    cw_pred  = (cw_prob  >= threshold).astype(int)
    rat_pred = (rat_prob >= threshold).astype(int)

    records = []
    for i, text in enumerate(texts):
        active_rats = [col for j, col in enumerate(LABEL_COLS) if rat_pred[i, j] == 1]
        if cw_pred[i] == 1 and active_rats:
            reasons = [RAT_DESCRIPTIONS[r] for r in active_rats]
            explanation = 'Check-worthy because this claim ' + ', and '.join(reasons) + '.'
        elif cw_pred[i] == 1:
            explanation = 'Check-worthy (no specific rationality label activated above threshold).'
        else:
            explanation = 'Not check-worthy.'

        rec = {
            'claim':        text,
            'check_worthy': int(cw_pred[i]),
            'cw_prob':      round(float(cw_prob[i]), 4),
            'explanation':  explanation,
        }
        for j, lbl in enumerate(LABEL_COLS):
            rec[lbl]           = int(rat_pred[i, j])
            rec[f'{lbl}_prob'] = round(float(rat_prob[i, j]), 4)
        records.append(rec)

    return pd.DataFrame(records)


demo = [
    "Drinking bleach can cure COVID-19.",
    "Approximately 40% of the global population uses Twitter.",
    "Critical health crisis ignored by government officials! They're hiding the truth.",
    "I like eating oranges, it contains Vitamin-C.",
    "Skipping vaccines is the key to natural immunity. Say no to vaccines!",
    "No single VIRUS can ever wipe out the entire human race from this earth.",
]

results = predict(demo, model, tokenizer)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 300)

print('\n' + '='*75)
print('  INFERENCE RESULTS – SUMMARY')
print('='*75)
print(results[['claim', 'check_worthy', 'cw_prob'] + LABEL_COLS].to_string(index=False))

print('\n' + '='*75)
print('  EXPLANATIONS')
print('='*75)
for _, row in results.iterrows():
    print(f'  CLAIM : {row["claim"][:72]}')
    print(f'  RESULT: CW={row["check_worthy"]} (p={row["cw_prob"]:.3f})')
    print(f'  WHY   : {row["explanation"]}')
    print()

## 16. Save Final Model

In [ ]:
final_path = os.path.join(OUTPUT_DIR, 'checkmate_mulcon_final.pt')
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'bert_model_name': BERT_MODEL_NAME,
        'num_rat_labels':  NUM_RAT_LABELS,
        'label_cols':      LABEL_COLS,
        'embed_dim':       EMBED_DIM,
        'label_embed_dim': LABEL_EMBED_DIM,
        'proj_dim':        PROJ_DIM,
        'num_attn_heads':  NUM_ATTN_HEADS,
        'dropout':         DROPOUT,
        'threshold':       THRESHOLD,
        'agg_mode':        'weighted',
    },
    'test_metrics': {
        'cw_acc':       test_m['cw_acc'],
        'cw_macro_f1':  test_m['cw_macro_f1'],
        'rat_macro_f1': test_m['rat_macro_f1'],
    }
}, final_path)
print(f'Saved: {final_path}')

print('\n' + '='*55)
print('FINAL RESULTS')
print('='*55)
print(f'CW  Accuracy       : {test_m["cw_acc"]:.4f}')
print(f'CW  Macro F1       : {test_m["cw_macro_f1"]:.4f}')
print(f'CW  F1 (cw)        : {test_m["cw_f1_cw"]:.4f}')
print(f'CW  F1 (non-cw)    : {test_m["cw_f1_ncw"]:.4f}')
print(f'RAT Macro F1       : {test_m["rat_macro_f1"]:.4f}')
for lbl, f1 in zip(LABEL_COLS, test_m['rat_per_label']):
    print(f'  {lbl:<35}: {f1:.4f}')
print('='*55)

## Appendix – Reload Saved Model

In [ ]:
def load_model(path, device=DEVICE):
    ckpt = torch.load(path, map_location=device)
    cfg  = ckpt['config']
    m = CheckMateModel(
        bert_name  = cfg['bert_model_name'],
        num_labels = cfg['num_rat_labels'],
        embed_dim  = cfg['embed_dim'],
        label_dim  = cfg['label_embed_dim'],
        proj_dim   = cfg['proj_dim'],
        num_heads  = cfg['num_attn_heads'],
        dropout    = cfg['dropout'],
        agg_mode   = cfg.get('agg_mode', 'weighted')
    ).to(device)
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()
    print(f'Loaded | saved metrics: {ckpt["test_metrics"]}')
    return m

# loaded = load_model('./checkpoints/checkmate_mulcon_final.pt')
# results = predict(["Vaccines cause autism."], loaded, tokenizer)
print('Done.')